# Fine-tuning Médical — Phi-3.5-mini avec LoRA (QLoRA 4-bit)

> **Mission Expérimentale — TechCorp Hackathon**  
> Fine-tuning d'un modèle de langage sur des conversations médicales.  
> Modèle résultant : **expérimental uniquement**, pas pour usage clinique.

**Stack :** Unsloth · PEFT/LoRA · HuggingFace Transformers · TRL SFTTrainer  
**Dataset :** [ruslanmv/ai-medical-chatbot](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot)  
**Durée estimée :** ~30–45 min sur GPU T4 (Colab)

---
⚠️ **Disclaimer médical** : Ce modèle est expérimental et ne remplace pas l'expertise d'un professionnel de santé qualifié.

In [ ]:
# Vérification GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU non détecté. Activez le GPU : Runtime > Change runtime type > T4 GPU")

gpu = torch.cuda.get_device_properties(0)
print(f"GPU         : {gpu.name}")
print(f"VRAM totale : {gpu.total_memory / 1e9:.1f} GB")
print(f"CUDA        : {torch.version.cuda}")

In [ ]:
# Installation des dépendances
# Unsloth = fine-tuning 2x plus rapide, 50% moins de VRAM
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets trl evaluate

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
MODEL_NAME      = "unsloth/Phi-3.5-mini-instruct"  # Optimisé Unsloth
DATASET_NAME    = "ruslanmv/ai-medical-chatbot"
NUM_EXAMPLES    = 2000   # Limite pour T4 (16 GB VRAM)
MAX_SEQ_LEN     = 2048
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
BATCH_SIZE      = 2
GRAD_ACCUM      = 4      # Effective batch = 8
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 1      # 1 epoch suffit sur T4
OUTPUT_DIR      = "./medical_lora_adapter"

SYSTEM_PROMPT = (
    "You are a knowledgeable medical assistant. "
    "Provide accurate, evidence-based medical information. "
    "Always recommend consulting a qualified healthcare professional "
    "for diagnosis and treatment decisions."
)

print("Configuration chargée.")
print(f"  Modèle     : {MODEL_NAME}")
print(f"  Dataset    : {DATASET_NAME} ({NUM_EXAMPLES} exemples)")
print(f"  LoRA       : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  Entraînement : {NUM_EPOCHS} epoch(s), lr={LEARNING_RATE}")

## 1. Chargement et préparation du dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

print(f"Chargement du dataset : {DATASET_NAME}...")
raw = load_dataset(DATASET_NAME, split="train")

print(f"Exemples totaux  : {len(raw):,}")
print(f"Colonnes         : {raw.column_names}")
print(f"\nAperçu des 2 premiers exemples :")
print("-" * 60)
for i in range(min(2, len(raw))):
    row = raw[i]
    # Détection automatique des colonnes
    q = row.get("Patient", row.get("input", row.get("question", "")))
    a = row.get("Doctor",  row.get("output", row.get("answer", "")))
    print(f"Patient : {str(q)[:120]}...")
    print(f"Doctor  : {str(a)[:120]}...")
    print("-" * 60)

In [ ]:
def format_example(example):
    """Convertit un exemple brut au format chat Phi-3.5."""
    # Détection des colonnes (plusieurs formats possibles)
    q = example.get("Patient", example.get("input",    example.get("question", "")))
    a = example.get("Doctor",  example.get("output",   example.get("answer",   "")))

    if not q or not a:
        return {"text": ""}

    text = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{str(q).strip()}<|end|>\n"
        f"<|assistant|>\n{str(a).strip()}<|end|>"
    )
    return {"text": text}


# Limitation + formatage
subset  = raw.select(range(min(NUM_EXAMPLES, len(raw))))
dataset = subset.map(format_example, num_proc=2)
dataset = dataset.filter(lambda x: len(x["text"]) > 100)

# Supprimer les colonnes originales
keep_cols = ["text"]
drop_cols = [c for c in dataset.column_names if c not in keep_cols]
dataset   = dataset.remove_columns(drop_cols)

# Découpage train / validation
split   = dataset.train_test_split(test_size=0.05, seed=42)
train_d = split["train"]
val_d   = split["test"]

print(f"Exemples d'entraînement : {len(train_d):,}")
print(f"Exemples de validation  : {len(val_d):,}")
print(f"\nExemple formaté :")
print(train_d[0]["text"][:400], "...")

## 2. Chargement du modèle de base (QLoRA 4-bit)

In [ ]:
from unsloth import FastLanguageModel

print(f"Chargement de {MODEL_NAME} en 4-bit...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    dtype        = None,      # Auto-detect (bfloat16 si supporté)
)

print("Modèle chargé.")
print(f"  Paramètres totaux : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
print("Application de LoRA...")

model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_R,
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = LORA_DROPOUT,
    target_modules  = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias                    = "none",
    use_gradient_checkpointing = "unsloth",
    random_state            = 42,
    use_rslora              = False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"  Paramètres entraînables : {trainable:,} ({100*trainable/total:.2f}% du total)")
print(f"  Gain mémoire LoRA       : {100*(1 - trainable/total):.1f}% de réduction")

## 3. Entraînement

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    logging_steps               = 25,
    save_steps                  = 200,
    save_total_limit            = 2,
    eval_strategy               = "steps",
    eval_steps                  = 100,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    report_to                   = "none",
    seed                        = 42,
)

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_d,
    eval_dataset      = val_d,
    dataset_text_field = "text",
    max_seq_length    = MAX_SEQ_LEN,
    dataset_num_proc  = 2,
    packing           = False,
    args              = training_args,
)

steps_per_epoch = len(train_d) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Démarrage de l'entraînement...")
print(f"  Exemples    : {len(train_d):,}")
print(f"  Steps/epoch : ~{steps_per_epoch}")
print(f"  Steps total : ~{steps_per_epoch * NUM_EPOCHS}")

trainer_stats = trainer.train()

print(f"\nEntraînement terminé !")
print(f"  Loss finale      : {trainer_stats.training_loss:.4f}")
print(f"  Durée totale     : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Tokens/s         : {trainer_stats.metrics.get('train_tokens_per_second', 'N/A')}")

## 4. Évaluation qualitative

In [ ]:
FastLanguageModel.for_inference(model)  # Optimisation inférence Unsloth

def generate(question: str, max_new_tokens: int = 300) -> str:
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{question}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).replace("<|end|>", "").strip()


test_questions = [
    "What are the main symptoms of type 2 diabetes?",
    "How does hypertension affect the cardiovascular system?",
    "What is the difference between viral and bacterial pneumonia?",
    "Explain the mechanism of action of beta-blockers.",
    "What are the first-line treatments for major depressive disorder?",
]

print("=" * 65)
print("TEST DU MODÈLE MÉDICAL FINE-TUNÉ")
print("=" * 65)

for q in test_questions:
    print(f"\nQuestion : {q}")
    print(f"Réponse  : {generate(q)}")
    print("-" * 65)

In [ ]:
# Perplexité sur l'ensemble de validation
import math

eval_results = trainer.evaluate()
loss = eval_results.get("eval_loss", None)
if loss:
    print(f"Loss de validation  : {loss:.4f}")
    print(f"Perplexité          : {math.exp(loss):.2f}")
    print("(Plus bas = meilleur — bonne perplexité médicale : <10)")

## 5. Sauvegarde de l'adaptateur LoRA

In [ ]:
import os

# Sauvegarde locale (dans Colab)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

files_saved = os.listdir(OUTPUT_DIR)
print(f"Fichiers sauvegardés dans {OUTPUT_DIR} :")
for f in sorted(files_saved):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f:<40} {size:.1f} MB")

In [ ]:
# Téléchargement sur votre machine locale
import shutil
from google.colab import files

zip_name = "medical_lora_adapter"
shutil.make_archive(zip_name, "zip", OUTPUT_DIR)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e6
print(f"Archive : {zip_name}.zip ({zip_size:.0f} MB)")

files.download(f"{zip_name}.zip")
print("Téléchargement lancé...")

In [ ]:
# OPTIONNEL : Push vers Hugging Face Hub
# Décommentez et remplissez votre token HF

# from huggingface_hub import login
# HF_TOKEN = "hf_XXXXXXXXXXXX"  # Settings > Access Tokens sur huggingface.co
# login(token=HF_TOKEN)
#
# REPO_NAME = "votre-username/phi35-medical-lora"
# model.push_to_hub(REPO_NAME, token=HF_TOKEN)
# tokenizer.push_to_hub(REPO_NAME, token=HF_TOKEN)
# print(f"Modèle publié : https://huggingface.co/{REPO_NAME}")

print("Push HF désactivé (décommentez pour activer).")